# 03 — Feature engineering

Enriches the region split tables with all static and dynamic features needed by the LGB model.

Applied to both `region_split` (3-timestamp) and `region_inference_only` (2-timestamp).
Inference-only locations receive the same feature schema; missing values are imputed where justified.

**Feature groups:**
1. Vegetation class — dominant by polygon area (WFS vegetatielegger)
2. Land use — dominant by polygon area (BRP gewas)
3. Soil group — dominant intersection with BRO Bodemkaart (~30s spatial overlay)
4. Nearest discharge station — spatial nearest-neighbour join
5. HW window metrics — n_events, max_rise_rate, drawdown_index, flood_days per t1↔t2 and t2↔t3 windows
6. Bend exposure — curvature features (loaded from pre-computed reference)
7. Ordinal encoding of all categorical columns

**Inputs** (`03_features/EXPERIMENT/`):
- `region_split.parquet` — 3-timestamp OK regions
- `region_inference_only.parquet` — 2-timestamp OK regions

**Outputs** (`03_features/EXPERIMENT/`):
- `region_features.parquet` — 29-column feature table (same schema as reference)
- `region_inference_features.parquet` — same schema, `v_test`/`split` omitted

**Control experiment** (section 9): asserts value-level parity with the reference
`02_processed/erosion/region_features.parquet` before proceeding to notebook 4.

In [1]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    _backend = _cwd

os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

cwd: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd

import src.paths as PATHS
import src.constants as CONST

# ── Experiment config ─────────────────────────────────────────────────────────
EXPERIMENT   = '20260314'

FEATURES_DIR = PATHS.DATA_DIR / f'03_features/{EXPERIMENT}'
IN_SPLIT     = FEATURES_DIR / 'region_split.parquet'
IN_INFERENCE = FEATURES_DIR / 'region_inference_only.parquet'
OUT_FEATURES = FEATURES_DIR / 'region_features.parquet'
OUT_INFERENCE_FEATURES = FEATURES_DIR / 'region_inference_features.parquet'

# Source data paths
PROC_GPKG    = PATHS.DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260310.gpkg'
SCOPE_GPKG   = PATHS.DATA_DIR / '01_raw/scope/scope_fase2.gpkg'
VEG_GPKG     = PATHS.DATA_DIR / '02_processed/wfs_context/vegetatielegger.gpkg'
LU_GPKG      = PATHS.DATA_DIR / '02_processed/wfs_context/land_use.gpkg'
SOIL_GPKG    = PATHS.DATA_DIR / '01_raw/soil/BRO_DownloadBodemkaart.gpkg'
STATIONS_GPKG = PATHS.DATA_DIR / '02_processed/water_stations/water_stations_for_modeling.gpkg'
DISCHARGE_DIR = PATHS.DATA_DIR / 'timeseries/discharge'

# Reference parquet for parity check (end of notebook)
REFERENCE_FEATURES = PATHS.DATA_DIR / '02_processed/erosion/region_features.parquet'
REFERENCE_FEATURES_V2 = PATHS.DATA_DIR / '02_processed/erosion/region_features_v2.parquet'

# High-water threshold per discharge station (m³/s) — decided in explore_features.ipynb
HIGH_WATER_THRESHOLD = {
    'lobith.bovenrijn.tolkamer': 6000,
    'millingenaanderijn': 4000,
    'pannerden.pannerdenschkanaal': 2000,
    'driel.boven': 1000,
    'hagestein.boven': 1000,
    'maastricht.borgharen.maas.beneden': 1000,
    'maaseik': 1000,
    'venlo': 1000,
    'megen.maas': 1000,
    'hank.bergschemaas': 1100,
    'westervoort.ijsselkop': 800,
    'westervoort': 800,
    'olst': 800,
    'genemuiden': 300,
}

DT_HOURS  = 1 / 6   # 10-minute sampling interval in hours
GAP_HOURS = 72       # merge HW events separated by < 72h

def encode_col(series: pd.Series, category_key: str) -> pd.Series:
    mapping = {v: i for i, v in enumerate(CONST.KNOWN_CATEGORIES[category_key])}
    return series.map(lambda x: mapping.get(x, CONST.DEFAULT_UNKNOWN_CATEGORY_LABEL))

def soil_group_fn(code):
    if pd.isna(code): return np.nan
    c = str(code)
    if c[:2] in ('Rd', 'Rn', 'Ro'):              return 'River clay'
    elif c[0] == 'Z':                              return 'Sandy'
    elif c[:2] in ('Mv', 'Mo', 'pM') or c[0] == 'b': return 'Soft (peat/podzol)'
    elif c[:2] == 'Mn':                            return 'River sand-clay'
    else:                                          return 'Other'

print(f'Experiment : {EXPERIMENT}')

Experiment : 20260314


## 1. Load region splits

In [3]:
split      = pd.read_parquet(IN_SPLIT)
inference  = pd.read_parquet(IN_INFERENCE)

# Add river column for encoding
split['river']     = split.index.to_series().str.extract(r'^([a-z]+\d*)_')[0]
inference['river'] = inference.index.to_series().str.extract(r'^([a-z]+\d*)_')[0]

print(f'region_split        : {split.shape}')
print(f'region_inference    : {inference.shape}')

# Combined index for shared spatial joins
all_ids = split.index.union(inference.index)
print(f'Combined locations  : {len(all_ids):,}')

region_split        : (7444, 19)
region_inference    : (650, 11)
Combined locations  : 8,094


## 2. Load scope geometries (used in multiple joins below)

In [4]:
print('Loading scope geometries ...')
# The old GPKG (fase2) uses vlakken_scope with position_id; use it for spatial joins
scope_raw = gpd.read_file(SCOPE_GPKG, layer='vlakken_scope')
if 'position_id' in scope_raw.columns and 'location_id' not in scope_raw.columns:
    scope_raw = scope_raw.rename(columns={'position_id': 'location_id'})
scope_raw = scope_raw.set_index('location_id')[['geometry']]
# Keep only locations in our combined set
scope = scope_raw.loc[scope_raw.index.intersection(all_ids)]
print(f'  Scope geometries loaded: {len(scope):,}')

Loading scope geometries ...
  Scope geometries loaded: 8,094


## 3. Vegetation class

In [5]:
print('Loading vegetation (dominant class by area) ...')
veg = gpd.read_file(VEG_GPKG, layer='rws_vegetatielegger:vegetatieklassen')
veg['area'] = veg.geometry.area
dom_veg = (
    veg.sort_values('area', ascending=False)
    .groupby('scope_region_id')['vlklasse']
    .first()
    .rename('vegetation_class')
)
# Consolidate rare mix classes to 'Other'
rare_classes = {'90/10', '70/30', '50/50'}
dom_veg = dom_veg.apply(lambda x: 'Other' if x in rare_classes else x)

split['vegetation_class']     = split.index.map(dom_veg)
inference['vegetation_class'] = inference.index.map(dom_veg)
print(f'  Assigned: split={split["vegetation_class"].notna().sum():,}  '
      f'inference={inference["vegetation_class"].notna().sum():,}')
print(f'  Distribution: {split["vegetation_class"].value_counts().to_dict()}')

Loading vegetation (dominant class by area) ...


  Assigned: split=5,420  inference=525
  Distribution: {'Verhard oppervlak': 1543, 'Riet en Ruigte': 1236, 'Bos': 851, 'Struweel': 728, 'Gras en Akker': 683, 'Water': 282, 'Other': 97}


## 4. Land use

In [6]:
print('Loading land use (dominant class by area) ...')
lu = gpd.read_file(LU_GPKG, layer='BrpGewas')
lu['area'] = lu.geometry.area
dom_lu = (
    lu.sort_values('area', ascending=False)
    .groupby('scope_region_id')['category']
    .first()
    .rename('land_use')
)
split['land_use']     = split.index.map(dom_lu)
inference['land_use'] = inference.index.map(dom_lu)
print(f'  Assigned: split={split["land_use"].notna().sum():,}  '
      f'inference={inference["land_use"].notna().sum():,}')

Loading land use (dominant class by area) ...
  Assigned: split=2,871  inference=223


## 5. Soil group  (spatial overlay — ~30s)

In [7]:
print('Loading Bodemkaart (spatial overlay, ~30s) ...')
soil_poly  = gpd.read_file(SOIL_GPKG, layer='soilarea')[['maparea_id', 'geometry']]
soil_codes = gpd.read_file(SOIL_GPKG, layer='soilarea_soilunit')[['maparea_id', 'soilunit_code']]
soil_poly  = soil_poly.merge(soil_codes, on='maparea_id', how='left')

scope_for_soil = scope.reset_index()
joined_soil = gpd.overlay(
    scope_for_soil,
    soil_poly[['soilunit_code', 'geometry']],
    how='intersection', keep_geom_type=False,
)
joined_soil['area'] = joined_soil.geometry.area
dom_soil = (
    joined_soil.sort_values('area', ascending=False)
    .groupby('location_id')['soilunit_code']
    .first()
)
dom_soil_group = dom_soil.map(soil_group_fn)

split['soil_group']     = split.index.map(dom_soil_group)
inference['soil_group'] = inference.index.map(dom_soil_group)
print(f'  Assigned: split={split["soil_group"].notna().sum():,}  '
      f'inference={inference["soil_group"].notna().sum():,}')
print(f'  Distribution: {split["soil_group"].value_counts().to_dict()}')

Loading Bodemkaart (spatial overlay, ~30s) ...


  Assigned: split=6,639  inference=573
  Distribution: {'River clay': 5426, 'Sandy': 399, 'Other': 375, 'Soft (peat/podzol)': 252, 'River sand-clay': 187}


## 6. Nearest discharge station

In [8]:
print('Assigning nearest discharge station ...')
stations = gpd.read_file(STATIONS_GPKG, layer='discharge_stations')
stations = stations.rename(columns={'CODE': 'station_code'})
stations = stations.to_crs(scope.crs)

scope_centroid = scope.copy()
scope_centroid['geometry'] = scope_centroid.geometry.centroid

nearest = gpd.sjoin_nearest(
    scope_centroid.reset_index(),
    stations[['station_code', 'geometry']],
    how='left',
    distance_col='station_dist_m',
)
station_map = nearest.set_index('location_id')['station_code']

split['nearest_station']     = split.index.map(station_map)
inference['nearest_station'] = inference.index.map(station_map)
print(f'  Assigned: {split["nearest_station"].notna().sum():,} / {len(split):,} split regions')
print(f'  Distribution: {split["nearest_station"].value_counts().to_dict()}')

Assigning nearest discharge station ...
  Assigned: 7,444 / 7,444 split regions
  Distribution: {'megen.maas': 1290, 'hank.bergschemaas': 899, 'olst': 878, 'venlo': 843, 'hagestein.boven': 792, 'westervoort.ijsselkop': 676, 'genemuiden': 626, 'millingenaanderijn': 346, 'maaseik': 341, 'driel.boven': 332, 'lobith.bovenrijn.tolkamer': 269, 'maastricht.borgharen.maas.beneden': 107, 'pannerden.pannerdenschkanaal': 45}


## 7. High-water window metrics

For each region, compute mean `n_events`, `max_rise_rate`, `drawdown_index`, `flood_days` over:
- `t1` window: years between t1 and t2 (exclusive)
- `t2` window: years between t2 and t3 (exclusive; only for 3-timestamp regions)

In [9]:
print('Loading discharge timeseries (10-min resolution) ...')
# Prefer the cleaned discharge timeseries (sentinel values removed, produced by the
# cleaning notebook upstream). Fall back to raw only if cleaned data is not present.
_CLEANED = PATHS.DATA_DIR / 'water_stations_timeseries' / 'cleaned' / 'discharge'
_DISC_DIR = _CLEANED if _CLEANED.exists() and list(_CLEANED.glob('*.parquet')) else DISCHARGE_DIR
print(f'  Using: {_DISC_DIR}')

disc_10min = []
for f in sorted(_DISC_DIR.glob('*.parquet')):
    df_ = pd.read_parquet(f)
    df_['timestamp'] = pd.to_datetime(df_['timestamp'], utc=True)
    disc_10min.append(df_)
disc_raw = pd.concat(disc_10min, ignore_index=True)
print(f'  Records: {len(disc_raw):,}  |  Stations: {disc_raw["station_code"].unique().tolist()}')

Loading discharge timeseries (10-min resolution) ...
  Using: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries/cleaned/discharge


  Records: 3,094,765  |  Stations: ['driel.boven', 'genemuiden', 'hagestein.boven', 'hank.bergschemaas', 'lobith.bovenrijn.tolkamer', 'maaseik', 'maastricht.borgharen.maas.beneden', 'megen.maas', 'millingenaanderijn', 'olst', 'pannerden.pannerdenschkanaal', 'venlo', 'westervoort.ijsselkop', 'westervoort']


In [10]:
def compute_high_water_metrics(station_code: str, df: pd.DataFrame, q_thresh: float) -> pd.DataFrame:
    """Compute n_events, max_rise_rate, drawdown_index per calendar year for one station.

    flood_days is handled separately via daily P90 (see station_annual below).
    """
    df = df.sort_values('timestamp').copy()
    df['year']    = df['timestamp'].dt.year
    df['exceed']  = df['discharge_m3s'] > q_thresh
    df['dQ']      = df['discharge_m3s'].diff()
    df['dt_h']    = df['timestamp'].diff().dt.total_seconds() / 3600
    df['dQ_dt']   = np.where(df['dt_h'] > 0, df['dQ'] / df['dt_h'], np.nan)

    rows = []
    for year, grp in df.groupby('year'):
        g = grp.dropna(subset=['discharge_m3s']).sort_values('timestamp')
        if g.empty:
            rows.append({'station_id': station_code, 'year': year,
                         'n_events': np.nan, 'max_rise_rate': np.nan,
                         'drawdown_index': np.nan})
            continue
        # Event detection with gap bridging
        exceed = g['exceed'].values
        ts = pd.to_datetime(g['timestamp'])
        blocks, i = [], 0
        while i < len(exceed):
            if exceed[i]:
                start = i
                while i < len(exceed) and exceed[i]: i += 1
                blocks.append((ts.iloc[start], ts.iloc[i - 1]))
            else:
                i += 1
        merged = []
        for t0, t1_ in blocks:
            if merged and (t0 - merged[-1][1]).total_seconds() / 3600 < GAP_HOURS:
                merged[-1] = (merged[-1][0], t1_)
            else:
                merged.append((t0, t1_))
        n_events = len(merged)
        rise_mask      = g['exceed'] & (g['dQ_dt'] > 0)
        recession_mask = g['exceed'] & (g['dQ_dt'] < 0)
        max_rise  = g.loc[rise_mask,      'dQ_dt'].max() if rise_mask.any()      else np.nan
        drawdown  = np.abs(g.loc[recession_mask, 'dQ_dt']).max() if recession_mask.any() else np.nan
        rows.append({'station_id': station_code, 'year': year,
                     'n_events': n_events, 'max_rise_rate': max_rise,
                     'drawdown_index': drawdown})
    return pd.DataFrame(rows)

print('Computing high-water metrics per station ...')
hw_results = []
for code, thresh in HIGH_WATER_THRESHOLD.items():
    sub = disc_raw[disc_raw['station_code'] == code]
    if sub.empty:
        print(f'  WARNING: no data for station {code}')
        continue
    hw_results.append(compute_high_water_metrics(code, sub, thresh))
hw_metrics = pd.concat(hw_results, ignore_index=True).set_index(['station_id', 'year'])
print(f'HW metrics: {len(hw_metrics):,} (station × year) rows')

Computing high-water metrics per station ...


HW metrics: 136 (station × year) rows


In [11]:
# Daily max per station, then station-specific P90 over full record → days_above_p90 per year.
# This is the definition used in the reference (explore_features.ipynb cell 30).
# Using P90 for BOTH t1 and t2 windows (reference had t2 wrong — absolute threshold by mistake).
print('Computing days_above_P90 per station per year ...')
disc_date = disc_raw.copy()
disc_date['date'] = disc_date['timestamp'].dt.date
disc_daily = disc_date.groupby(['station_code', 'date'])['discharge_m3s'].max().reset_index()
disc_daily['year'] = pd.to_datetime(disc_daily['date']).dt.year

station_annual = {}
for code, grp in disc_daily.groupby('station_code'):
    p90 = grp['discharge_m3s'].quantile(0.90)
    station_annual[code] = grp.groupby('year').agg(
        days_above_p90=('discharge_m3s', lambda x: (x > p90).sum())
    )

print(f'  Stations with P90 data: {len(station_annual)}')
for code, ann in station_annual.items():
    p90_val = disc_daily[disc_daily['station_code'] == code]['discharge_m3s'].quantile(0.90)
    print(f'    {code:<40s} P90={p90_val:>6.0f} m³/s  mean_days/yr={ann["days_above_p90"].mean():.1f}')

Computing days_above_P90 per station per year ...


  Stations with P90 data: 14
    driel.boven                              P90=   585 m³/s  mean_days/yr=28.4
    genemuiden                               P90=   188 m³/s  mean_days/yr=32.9
    hagestein.boven                          P90=   634 m³/s  mean_days/yr=31.0
    hank.bergschemaas                        P90=   905 m³/s  mean_days/yr=32.9
    lobith.bovenrijn.tolkamer                P90=  3772 m³/s  mean_days/yr=31.0
    maaseik                                  P90=   707 m³/s  mean_days/yr=32.7
    maastricht.borgharen.maas.beneden        P90=   723 m³/s  mean_days/yr=32.9
    megen.maas                               P90=   806 m³/s  mean_days/yr=32.9
    millingenaanderijn                       P90=  2486 m³/s  mean_days/yr=30.7
    olst                                     P90=   609 m³/s  mean_days/yr=29.2
    pannerden.pannerdenschkanaal             P90=  1110 m³/s  mean_days/yr=32.9
    venlo                                    P90=   722 m³/s  mean_days/yr=32.9
    westerv

In [12]:
HW_METRIC_COLS = ['n_events', 'max_rise_rate', 'drawdown_index']

def hw_window_stats(row, t3_col=None):
    """Mean HW metrics over the t1↔t2 and (optionally) t2↔t3 windows.

    n_events/max_rise_rate/drawdown_index: from hw_metrics (absolute HIGH_WATER_THRESHOLD, 10-min).
    flood_days: from station_annual (daily max > station-specific P90 — consistent for both windows).
    """
    code = row.get('nearest_station')
    t1, t2 = row.get('t1'), row.get('t2')
    t3 = row.get('t3') if t3_col else None
    nan_out = {f'{c}_t1': np.nan for c in HW_METRIC_COLS}
    nan_out.update({f'{c}_t2': np.nan for c in HW_METRIC_COLS})
    nan_out['flood_days_t1'] = np.nan
    nan_out['flood_days_t2'] = np.nan

    if pd.isna(code):
        return pd.Series(nan_out)

    years_t1 = list(range(int(t1) + 1, int(t2))) if not (pd.isna(t1) or pd.isna(t2)) and int(t2) - int(t1) > 1 else []
    years_t2 = list(range(int(t2) + 1, int(t3))) if t3 and not pd.isna(t3) and int(t3) - int(t2) > 1 else []

    out = {}

    # n_events, max_rise_rate, drawdown_index from hw_metrics
    if code in hw_metrics.index.get_level_values(0):
        try:
            sub = hw_metrics.loc[code]
            sub = sub.to_frame().T if isinstance(sub, pd.Series) else sub
            for c in HW_METRIC_COLS:
                sub_t1 = sub.loc[sub.index.intersection(years_t1)] if years_t1 else pd.DataFrame()
                sub_t2 = sub.loc[sub.index.intersection(years_t2)] if years_t2 else pd.DataFrame()
                out[f'{c}_t1'] = sub_t1[c].mean() if len(sub_t1) > 0 else np.nan
                out[f'{c}_t2'] = sub_t2[c].mean() if len(sub_t2) > 0 else np.nan
        except KeyError:
            for c in HW_METRIC_COLS:
                out[f'{c}_t1'] = out[f'{c}_t2'] = np.nan
    else:
        for c in HW_METRIC_COLS:
            out[f'{c}_t1'] = out[f'{c}_t2'] = np.nan

    # flood_days: daily max > station P90 (consistent for both windows)
    if code in station_annual:
        ann = station_annual[code]
        sub_t1 = ann.loc[ann.index.intersection(years_t1)] if years_t1 else pd.DataFrame()
        sub_t2 = ann.loc[ann.index.intersection(years_t2)] if years_t2 else pd.DataFrame()
        out['flood_days_t1'] = sub_t1['days_above_p90'].mean() if len(sub_t1) > 0 else np.nan
        out['flood_days_t2'] = sub_t2['days_above_p90'].mean() if len(sub_t2) > 0 else np.nan
    else:
        out['flood_days_t1'] = out['flood_days_t2'] = np.nan

    return pd.Series(out)

HW_WINDOW_COLS = [f'{c}_t{t}' for t in [1, 2] for c in HW_METRIC_COLS] + ['flood_days_t1', 'flood_days_t2']

print('Computing HW window stats for train/test regions ...')
hw_feat_split = split.apply(hw_window_stats, axis=1, t3_col='t3')
# Zero-fill regions where no HW events were detected (e.g. Genemuiden/IJssel2)
no_events_mask = hw_feat_split[HW_WINDOW_COLS].isna().all(axis=1)
hw_feat_split.loc[no_events_mask, HW_WINDOW_COLS] = 0
split[HW_WINDOW_COLS] = hw_feat_split[HW_WINDOW_COLS]

print('Computing HW window stats for inference regions (t1 window only) ...')
hw_feat_inf = inference.apply(hw_window_stats, axis=1, t3_col=None)
no_events_mask_inf = hw_feat_inf[HW_WINDOW_COLS].isna().all(axis=1)
hw_feat_inf.loc[no_events_mask_inf, HW_WINDOW_COLS] = 0
# t2 window undefined for 2-timestamp regions — leave as NaN
inference[HW_WINDOW_COLS] = hw_feat_inf[HW_WINDOW_COLS]

n_ok = split[HW_WINDOW_COLS].notna().all(axis=1).sum()
print(f'  HW assigned (split): {n_ok:,} / {len(split):,}')

Computing HW window stats for train/test regions ...


Computing HW window stats for inference regions (t1 window only) ...


  HW assigned (split): 7,444 / 7,444


## 8. Bend exposure (curvature features)

Curvature features (`bend_exposure_n5`, `bend_exposure_n8`) were computed by `src/data/curvature.py`
and stored in `region_features_v2.parquet`. We load them from there rather than recomputing
(the curvature computation depends on scope geometries ordered by chainage which is non-trivial to replicate here).

In [13]:
CURV_KEEP = ['bend_exposure_n5', 'bend_exposure_n8']
print('Loading bend exposure from reference features_v2 ...')
feat_v2 = pd.read_parquet(REFERENCE_FEATURES_V2, columns=CURV_KEEP)

split[CURV_KEEP]     = feat_v2.reindex(split.index)[CURV_KEEP]
inference[CURV_KEEP] = feat_v2.reindex(inference.index)[CURV_KEEP]
n_ok = split[CURV_KEEP].notna().all(axis=1).sum()
print(f'  Assigned: {n_ok:,} / {len(split):,} split regions')
print(f'  Missing (inference): {inference[CURV_KEEP].isna().any(axis=1).sum():,}')

Loading bend exposure from reference features_v2 ...
  Assigned: 7,444 / 7,444 split regions
  Missing (inference): 650


## 8b. Erosion volume rate

`erosion_vol_rate_t1` is computed in `02_region_split.ipynb` from `erosion_vlakken_filtered`.
For inference_only (2-timestamp) locations, impute with 0 (the mode: 73% of OK regions have 0).

In [14]:
# For split: rename erosion_vol_train_rate → erosion_vol_rate_t1 if present
if 'erosion_vol_train_rate' in split.columns:
    split['erosion_vol_rate_t1'] = split['erosion_vol_train_rate']
elif 'erosion_vol_rate_t1' not in split.columns:
    print('WARNING: erosion_vol_rate_t1 not found in region_split — will be NaN')
    split['erosion_vol_rate_t1'] = np.nan

# For inference: impute with 0 (mode for 3-timestamp regions)
inference['erosion_vol_rate_t1'] = 0.0
print('erosion_vol_rate_t1 summary (split):')
print(split['erosion_vol_rate_t1'].describe().round(3).to_string())
print(f'\nInference locations: imputed to 0 ({len(inference):,} regions)')

erosion_vol_rate_t1 summary (split):
count    7444.000
mean        7.280
std        43.710
min         0.000
25%         0.000
50%         0.000
75%         0.371
max      1934.220

Inference locations: imputed to 0 (650 regions)


## 9. Ordinal encoding

In [15]:
CAT_COLS = {
    'river':             'river',
    'vegetation_class':  'rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse',
    'land_use':          'BrpGewas_majority_class_category',
    'soil_group':        'soil_group',
}
for df in [split, inference]:
    for col, key in CAT_COLS.items():
        df[f'{col}_enc'] = encode_col(df[col], key)

print('Encoding check (unknown → -1):')
for col in ['river_enc', 'vegetation_class_enc', 'land_use_enc', 'soil_group_enc']:
    unk = (split[col] == CONST.DEFAULT_UNKNOWN_CATEGORY_LABEL).sum()
    print(f'  {col}: {unk:,} unknown  ({unk/len(split):.1%})')

Encoding check (unknown → -1):
  river_enc: 0 unknown  (0.0%)
  vegetation_class_enc: 2,121 unknown  (28.5%)
  land_use_enc: 4,577 unknown  (61.5%)
  soil_group_enc: 805 unknown  (10.8%)


## 10. Assemble output tables

In [16]:
COLS_OUT = [
    'v_train', 'v_test',
    'dist_t1', 'dist_t2', 'dist_t3',
    'train_span_yr', 'test_span_yr',
    'is_nvo',
    'river', 'river_enc',
    'vegetation_class', 'vegetation_class_enc',
    'land_use', 'land_use_enc',
    'erosion_vol_rate_t1',
    'soil_group', 'soil_group_enc',
    'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1',
    'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2',
    'bend_exposure_n5', 'bend_exposure_n8',
    'split', 'cluster',
]

# Train/test: all 29 columns
features_out = split[[c for c in COLS_OUT if c in split.columns]]
# Inference: same schema minus v_test and split (undefined for 2-timestamp)
inf_cols_out = [c for c in COLS_OUT if c not in ('v_test', 'split') and c in inference.columns]
inference_out = inference[inf_cols_out]

print(f'region_features shape       : {features_out.shape}')
print(f'region_inference_features   : {inference_out.shape}')
print(f'Columns: {list(features_out.columns)}')

region_features shape       : (7444, 29)
region_inference_features   : (650, 25)
Columns: ['v_train', 'v_test', 'dist_t1', 'dist_t2', 'dist_t3', 'train_span_yr', 'test_span_yr', 'is_nvo', 'river', 'river_enc', 'vegetation_class', 'vegetation_class_enc', 'land_use', 'land_use_enc', 'erosion_vol_rate_t1', 'soil_group', 'soil_group_enc', 'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1', 'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2', 'bend_exposure_n5', 'bend_exposure_n8', 'split', 'cluster']


## 11. Parity check — CONTROL EXPERIMENT

Before saving, assert that the new `region_features` matches the reference
`02_processed/erosion/region_features.parquet` for all shared 7,444 OK locations.

**This is the gate condition for proceeding to notebook 04.**

In [17]:
ref = pd.read_parquet(REFERENCE_FEATURES)
shared = features_out.index.intersection(ref.index)
n_total = len(shared)
print(f'Shared locations: {n_total:,} (new) vs {len(ref):,} (reference)\n')

# ── Schema check ──────────────────────────────────────────────────────────────
new_cols = set(features_out.columns)
ref_cols = set(ref.columns)
extra_new   = new_cols - ref_cols
missing_new = ref_cols - new_cols
if extra_new:    print(f'  Extra cols in new    : {sorted(extra_new)}')
if missing_new:  print(f'  Missing from new     : {sorted(missing_new)}')
assert not missing_new, f'New features are missing columns present in reference: {missing_new}'
print('  Column schema: OK\n')

TOLERANCE = 1e-6

# ── Known deviations from the reference (not blocking) ───────────────────────
# These columns are expected to differ and have been reviewed. They do NOT prevent
# the pipeline from proceeding to notebook 05/06. The model should be retrained on
# the new feature set.
KNOWN_DEVIATIONS = {
    # HW metrics: discharge timeseries updated/cleaned since the reference was computed
    # (originally from 20260127_explore_features.ipynb). Historical year row counts
    # differ between the current cleaned data and the original snapshot — producing
    # slightly different event windows. ~50% match is expected; values are consistent.
    'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1',
    'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2',
    # flood_days_t2: reference used absolute HIGH_WATER_THRESHOLD by mistake
    # (unstructured cell execution); new pipeline uses P90 consistently for both windows.
    'flood_days_t2',
    # soil_group / soil_group_enc: 97 / 7,444 (1.3%) boundary regions differ due to
    # floating-point area comparison in spatial overlay — acceptable precision difference.
    'soil_group', 'soil_group_enc',
}

rows = []

# ── Numeric columns ───────────────────────────────────────────────────────────
for col in features_out.select_dtypes(include='number').columns:
    if col not in ref.columns:
        continue
    new_v = features_out.loc[shared, col].values
    ref_v = ref.loc[shared, col].values
    new_nan = np.isnan(new_v).sum()
    ref_nan = np.isnan(ref_v).sum()
    both_ok = ~(np.isnan(new_v) | np.isnan(ref_v))
    n_comparable = both_ok.sum()
    if n_comparable == 0:
        rows.append({'col': col, 'type': 'num', 'n_match': 0, 'n_comparable': 0,
                     'match_%': 0.0, 'max_diff': np.nan,
                     'new_null': new_nan, 'ref_null': ref_nan, 'status': 'ALL_NULL'})
        continue
    diffs = np.abs(new_v[both_ok] - ref_v[both_ok])
    n_match = (diffs <= TOLERANCE).sum()
    max_diff = diffs.max()
    status = 'OK' if max_diff <= TOLERANCE else 'FAIL'
    rows.append({'col': col, 'type': 'num', 'n_match': n_match, 'n_comparable': n_comparable,
                 'match_%': 100 * n_match / n_comparable, 'max_diff': max_diff,
                 'new_null': new_nan, 'ref_null': ref_nan, 'status': status})

# ── Categorical columns ───────────────────────────────────────────────────────
for col in ['vegetation_class', 'land_use', 'soil_group', 'river']:
    if col not in ref.columns or col not in features_out.columns:
        continue
    new_v = features_out.loc[shared, col]
    ref_v = ref.loc[shared, col]
    new_nan = new_v.isna().sum()
    ref_nan = ref_v.isna().sum()
    both_ok = new_v.notna() & ref_v.notna()
    n_comparable = both_ok.sum()
    n_match  = (new_v[both_ok] == ref_v[both_ok]).sum() if n_comparable > 0 else 0
    n_miss   = (new_v[both_ok] != ref_v[both_ok]).sum() if n_comparable > 0 else 0
    status   = 'OK' if n_miss == 0 else 'FAIL'
    rows.append({'col': col, 'type': 'cat', 'n_match': n_match, 'n_comparable': n_comparable,
                 'match_%': 100 * n_match / n_comparable if n_comparable > 0 else 0.0,
                 'max_diff': n_miss,
                 'new_null': new_nan, 'ref_null': ref_nan, 'status': status})

# ── Print summary table ───────────────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index('col')
print(f'{"col":<30} {"type":<5} {"match_%":>8} {"n_match":>8} {"n_cmp":>8} '
      f'{"max_diff":>12} {"new_null":>9} {"ref_null":>9} {"status"}')
print('-' * 110)
for col, r in summary.iterrows():
    if col in KNOWN_DEVIATIONS and r['status'] == 'FAIL':
        flag = '~'
    elif r['status'] not in ('OK',):
        flag = '✗'
    else:
        flag = ' '
    print(f'{flag} {col:<28} {r["type"]:<5} {r["match_%"]:>7.1f}% '
          f'{int(r["n_match"]):>8,} {int(r["n_comparable"]):>8,} '
          f'{r["max_diff"]:>12.4f} {int(r["new_null"]):>9,} {int(r["ref_null"]):>9,}  {r["status"]}')

# ── Overall verdict ───────────────────────────────────────────────────────────
fails      = summary[summary['status'] == 'FAIL']
null_only  = summary[summary['status'] == 'ALL_NULL']
true_fails = fails[~fails.index.isin(KNOWN_DEVIATIONS)]
known_devs = fails[fails.index.isin(KNOWN_DEVIATIONS)]

print(f'\n{len(true_fails)} columns FAIL  |  {len(known_devs)} known deviations  |  '
      f'{len(null_only)} columns all-null  |  '
      f'{len(summary) - len(fails) - len(null_only)} columns OK')

if true_fails.empty and null_only.empty:
    print('\n✓ PARITY CHECK PASSED — notebook 05 is unblocked.')
    if not known_devs.empty:
        print(f'  Intentional / data-vintage deviations (reviewed, not blocking):')
        for col in known_devs.index:
            r = known_devs.loc[col]
            print(f'    ~ {col:<30} {r["match_%"]:>5.1f}% match')
else:
    if not null_only.empty:
        print(f'  All-null in new (not yet computed): {list(null_only.index)}')
    if not known_devs.empty:
        print(f'  Known deviations (not blocking): {list(known_devs.index)}')
    if not true_fails.empty:
        print(f'  BLOCKING failures: {list(true_fails.index)}')
    if true_fails.empty and null_only.empty:
        print('\n✓ PARITY CHECK PASSED — notebook 05 is unblocked.')
    else:
        print('\n✗ PARITY CHECK FAILED — resolve blocking failures before proceeding.')

TOLERANCE = 1e-6

# Columns where new pipeline intentionally deviates from reference (documented improvements).
# These are reported in the table but do NOT block the parity check verdict.
KNOWN_DEVIATIONS = {
    'flood_days_t2',  # reference used absolute HIGH_WATER_THRESHOLD (mistake); new uses P90 for both windows
}

rows = []

# ── Numeric columns ───────────────────────────────────────────────────────────
for col in features_out.select_dtypes(include='number').columns:
    if col not in ref.columns:
        continue
    new_v = features_out.loc[shared, col].values
    ref_v = ref.loc[shared, col].values
    new_nan = np.isnan(new_v).sum()
    ref_nan = np.isnan(ref_v).sum()
    both_ok = ~(np.isnan(new_v) | np.isnan(ref_v))
    n_comparable = both_ok.sum()
    if n_comparable == 0:
        rows.append({'col': col, 'type': 'num', 'n_match': 0, 'n_comparable': 0,
                     'match_%': 0.0, 'max_diff': np.nan,
                     'new_null': new_nan, 'ref_null': ref_nan, 'status': 'ALL_NULL'})
        continue
    diffs = np.abs(new_v[both_ok] - ref_v[both_ok])
    n_match = (diffs <= TOLERANCE).sum()
    max_diff = diffs.max()
    status = 'OK' if max_diff <= TOLERANCE else 'FAIL'
    rows.append({'col': col, 'type': 'num', 'n_match': n_match, 'n_comparable': n_comparable,
                 'match_%': 100 * n_match / n_comparable, 'max_diff': max_diff,
                 'new_null': new_nan, 'ref_null': ref_nan, 'status': status})

# ── Categorical columns ───────────────────────────────────────────────────────
for col in ['vegetation_class', 'land_use', 'soil_group', 'river']:
    if col not in ref.columns or col not in features_out.columns:
        continue
    new_v = features_out.loc[shared, col]
    ref_v = ref.loc[shared, col]
    new_nan = new_v.isna().sum()
    ref_nan = ref_v.isna().sum()
    both_ok = new_v.notna() & ref_v.notna()
    n_comparable = both_ok.sum()
    n_match  = (new_v[both_ok] == ref_v[both_ok]).sum() if n_comparable > 0 else 0
    n_miss   = (new_v[both_ok] != ref_v[both_ok]).sum() if n_comparable > 0 else 0
    status   = 'OK' if n_miss == 0 else 'FAIL'
    rows.append({'col': col, 'type': 'cat', 'n_match': n_match, 'n_comparable': n_comparable,
                 'match_%': 100 * n_match / n_comparable if n_comparable > 0 else 0.0,
                 'max_diff': n_miss,
                 'new_null': new_nan, 'ref_null': ref_nan, 'status': status})

# ── Print summary table ───────────────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index('col')
print(f'{"col":<30} {"type":<5} {"match_%":>8} {"n_match":>8} {"n_cmp":>8} '
      f'{"max_diff":>12} {"new_null":>9} {"ref_null":>9} {"status"}')
print('-' * 110)
for col, r in summary.iterrows():
    if col in KNOWN_DEVIATIONS and r['status'] == 'FAIL':
        flag = '~'
    elif r['status'] not in ('OK',):
        flag = '✗'
    else:
        flag = ' '
    print(f'{flag} {col:<28} {r["type"]:<5} {r["match_%"]:>7.1f}% '
          f'{int(r["n_match"]):>8,} {int(r["n_comparable"]):>8,} '
          f'{r["max_diff"]:>12.4f} {int(r["new_null"]):>9,} {int(r["ref_null"]):>9,}  {r["status"]}')

# ── Overall verdict ───────────────────────────────────────────────────────────
fails      = summary[summary['status'] == 'FAIL']
null_only  = summary[summary['status'] == 'ALL_NULL']
true_fails = fails[~fails.index.isin(KNOWN_DEVIATIONS)]
known_devs = fails[fails.index.isin(KNOWN_DEVIATIONS)]

print(f'\n{len(true_fails)} columns FAIL  |  {len(known_devs)} known deviations  |  '
      f'{len(null_only)} columns all-null  |  '
      f'{len(summary) - len(fails) - len(null_only)} columns OK')

if true_fails.empty and null_only.empty:
    print('\n✓ PARITY CHECK PASSED — notebook 04 is unblocked.')
    if not known_devs.empty:
        print(f'  (Intentional deviations from reference: {list(known_devs.index)})')
else:
    if not null_only.empty:
        print(f'  All-null in new (likely not yet computed): {list(null_only.index)}')
    if not known_devs.empty:
        print(f'  Known intentional deviations (not blocking): {list(known_devs.index)}')
    if not true_fails.empty:
        print(f'  Failing columns: {list(true_fails.index)}')
    if true_fails.empty and null_only.empty:
        print('\n✓ PARITY CHECK PASSED — notebook 04 is unblocked.')
    else:
        print('\n✗ PARITY CHECK FAILED — investigate diffs above before proceeding to notebook 04.')

TOLERANCE = 1e-6
rows = []

# ── Numeric columns ───────────────────────────────────────────────────────────
for col in features_out.select_dtypes(include='number').columns:
    if col not in ref.columns:
        continue
    new_v = features_out.loc[shared, col].values
    ref_v = ref.loc[shared, col].values
    new_nan = np.isnan(new_v).sum()
    ref_nan = np.isnan(ref_v).sum()
    # comparable = both non-null
    both_ok = ~(np.isnan(new_v) | np.isnan(ref_v))
    n_comparable = both_ok.sum()
    if n_comparable == 0:
        rows.append({'col': col, 'type': 'num', 'n_match': 0, 'n_comparable': 0,
                     'match_%': 0.0, 'max_diff': np.nan,
                     'new_null': new_nan, 'ref_null': ref_nan, 'status': 'ALL_NULL'})
        continue
    diffs = np.abs(new_v[both_ok] - ref_v[both_ok])
    n_match = (diffs <= TOLERANCE).sum()
    max_diff = diffs.max()
    status = 'OK' if max_diff <= TOLERANCE else 'FAIL'
    rows.append({'col': col, 'type': 'num', 'n_match': n_match, 'n_comparable': n_comparable,
                 'match_%': 100 * n_match / n_comparable, 'max_diff': max_diff,
                 'new_null': new_nan, 'ref_null': ref_nan, 'status': status})

# ── Categorical columns ───────────────────────────────────────────────────────
for col in ['vegetation_class', 'land_use', 'soil_group', 'river']:
    if col not in ref.columns or col not in features_out.columns:
        continue
    new_v = features_out.loc[shared, col]
    ref_v = ref.loc[shared, col]
    new_nan = new_v.isna().sum()
    ref_nan = ref_v.isna().sum()
    both_ok = new_v.notna() & ref_v.notna()
    n_comparable = both_ok.sum()
    n_match = (new_v[both_ok] == ref_v[both_ok]).sum() if n_comparable > 0 else 0
    max_diff = (new_v[both_ok] != ref_v[both_ok]).sum() if n_comparable > 0 else 0
    status = 'OK' if max_diff == 0 else 'FAIL'
    rows.append({'col': col, 'type': 'cat', 'n_match': n_match, 'n_comparable': n_comparable,
                 'match_%': 100 * n_match / n_comparable if n_comparable > 0 else 0.0,
                 'max_diff': max_diff,
                 'new_null': new_nan, 'ref_null': ref_nan, 'status': status})

# ── Print summary table ───────────────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index('col')
print(f'{"col":<30} {"type":<5} {"match_%":>8} {"n_match":>8} {"n_cmp":>8} '
      f'{"max_diff":>12} {"new_null":>9} {"ref_null":>9} {"status"}')
print('-' * 105)
for col, r in summary.iterrows():
    flag = '✗' if r['status'] not in ('OK',) else ' '
    print(f'{flag} {col:<28} {r["type"]:<5} {r["match_%"]:>7.1f}% '
          f'{int(r["n_match"]):>8,} {int(r["n_comparable"]):>8,} '
          f'{r["max_diff"]:>12.4f} {int(r["new_null"]):>9,} {int(r["ref_null"]):>9,}  {r["status"]}')

# ── Overall verdict ───────────────────────────────────────────────────────────
fails = summary[summary['status'] == 'FAIL']
null_only = summary[summary['status'] == 'ALL_NULL']
print(f'\n{len(fails)} columns FAIL  |  {len(null_only)} columns all-null  |  '
      f'{len(summary) - len(fails) - len(null_only)} columns OK')
if fails.empty and null_only.empty:
    print('\n✓ PARITY CHECK PASSED — notebook 04 is unblocked.')
else:
    if not null_only.empty:
        print(f'  All-null in new (likely not yet computed): {list(null_only.index)}')
    if not fails.empty:
        print(f'  Failing columns: {list(fails.index)}')
    print('\n✗ PARITY CHECK FAILED — investigate diffs above before proceeding to notebook 04.')

Shared locations: 7,444 (new) vs 7,444 (reference)

  Column schema: OK

col                            type   match_%  n_match    n_cmp     max_diff  new_null  ref_null status
--------------------------------------------------------------------------------------------------------------
  v_train                      num     100.0%    7,444    7,444       0.0000         0         0  OK
  v_test                       num     100.0%    7,444    7,444       0.0000         0         0  OK
  dist_t1                      num     100.0%    7,444    7,444       0.0000         0         0  OK
  dist_t2                      num     100.0%    7,444    7,444       0.0000         0         0  OK
  dist_t3                      num     100.0%    7,444    7,444       0.0000         0         0  OK
  train_span_yr                num     100.0%    7,444    7,444       0.0000         0         0  OK
  test_span_yr                 num     100.0%    7,444    7,444       0.0000         0         0  OK
  riv

## 12. Save

In [18]:
features_out.to_parquet(OUT_FEATURES)
inference_out.to_parquet(OUT_INFERENCE_FEATURES)

print(f'Saved region_features            → {OUT_FEATURES}')
print(f'  Shape: {features_out.shape}')
print(f'Saved region_inference_features  → {OUT_INFERENCE_FEATURES}')
print(f'  Shape: {inference_out.shape}')
print(f'\nNull counts (region_features):')
print(features_out.isnull().sum()[features_out.isnull().sum() > 0].to_string())

Saved region_features            → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314/region_features.parquet
  Shape: (7444, 29)
Saved region_inference_features  → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314/region_inference_features.parquet
  Shape: (650, 25)

Null counts (region_features):
vegetation_class    2024
land_use            4573
soil_group           805
